<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/3_dise%C3%B1o_entrenamiento_evaluacion/3_2_Generacion_X_y_con_filtrado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.2. Generación de X e y para modelos

## 0. Clonado de Repositorio, instalación de librería e importación.

### Clonado de Repositorio

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


### Acceso de Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Instalación de librerías

In [3]:
# !{sys.executable} -m pip install -q pandas_market_calendars  # Solo si usás horarios de mercados
!{sys.executable} -m pip install -q ta
print("Librerías instaladas: ta")

/bin/bash: line 1: {sys.executable}: command not found
Librerías instaladas: ta


### Importación de librerías

In [4]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Utilidades del sistema y fechas
import os
import glob
import requests
from datetime import datetime, timedelta

# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Análisis técnico
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

# Estadística
from scipy.stats import spearmanr

#que es?
#from tqdm.notebook import tqdm
from tqdm import tqdm
# Modelos ML
#from xgboost import XGBRegressor
#from sklearn.metrics import mean_squared_error, r2_score

# Calendario de mercados (descomentar si lo necesitás)
# import pandas_market_calendars as mcal

## 1. Carga de datasets train, valid y test.

In [5]:
def load_data_from_drive():
    """
    Función para cargar un archivo Parquet desde el drive
        """
    # Definir la URL del archivo Parquet en Drive
    df_path_mnq = f'{drive_path}/mnq_data/mnq_model.parquet'
    df_path_train = f'{drive_path}/mnq_data/mnq_train.parquet'
    df_path_valid = f'{drive_path}/mnq_data/mnq_valid.parquet'
    df_path_test = f'{drive_path}/mnq_data/mnq_test.parquet'
    df_path_factores = f'{drive_path}/df_factores.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [6]:
def load_data_from_repo():
    """
    Función para cargar un archivo Parquet desde el repositorio clonado
    """
    # Definir la URL del archivo Parquet en GitHub
    df_path_mnq = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_model.parquet'
    df_path_factores = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/df_factores.parquet'
    df_path_train = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_train.parquet'
    df_path_valid = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_valid.parquet'
    df_path_test = '/content/neural_profit/3_diseño_entrenamiento_evaluacion/mnq_test.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)
    df_factores = pd.read_parquet (df_path_factores)
    df_train = pd.read_parquet(df_path_train)
    df_valid = pd.read_parquet(df_path_valid)
    df_test = pd.read_parquet(df_path_test)

    return df_model, df_factores, df_train, df_valid, df_test

In [7]:
mnq_model, indicadores_tecnicos, mnq_train, mnq_valid, mnq_test = load_data_from_drive()

## 1.1. Información de los datasets

In [8]:
def info_dataset (df): # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"Cantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['target_return_30']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"Valores por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo
  print(f"Hora diaria de inicio {primer_hora}")
  print(f"Hora diaria de final {ultima_hora}")
  print(f"Zona horaria: {zona_horaria}")

In [9]:
info_dataset(mnq_train)

Cantidad de días: 917
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [10]:
info_dataset(mnq_valid)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


In [11]:
info_dataset(mnq_test)

Cantidad de días: 197
Valores por día: 361
Hora diaria de inicio 09:30
Hora diaria de final 15:30
Zona horaria: America/New_York


## 2. Generación de ventanas X e y

In [12]:
target_column = "target_return_30"
features = mnq_model.columns.tolist()
features.remove(target_column)
features.remove('date')

window_size = 60

In [13]:
#Función para generar ventanas y vectorizarlas
def generar_ventanas(df, features, target_col, window_size):
    X, y = [], []
    for fecha, grupo in tqdm(df.groupby("date"), desc="Procesando días"):
        grupo = grupo.reset_index(drop=True)
        for i in range(len(grupo) - window_size):
            ventana = grupo.loc[i:i+window_size-1, features]
            if ventana.isnull().any().any():
                continue
            vector = ventana.values.flatten()
            target = grupo.loc[i+window_size-1, target_col]
            X.append(vector)
            y.append(target)
    return np.array(X), np.array(y)

### Generamos los X_* e y_* con el total de features

Antes de correr la generación de X e y, tenemos que verificar si es que no existe en la carpeta ventanas_X_y:


In [14]:
# Subcarpeta donde querés guardar
save_dir = f"{drive_path}/ventanas_x_y"

# Crear carpeta si no existe
os.makedirs(save_dir, exist_ok=True)


In [15]:
#Ruta de x_y
ruta_x_y_train = f"{drive_path}/ventanas_x_y/mnq_Xy_train.npz"
ruta_x_y_valid = f"{drive_path}/ventanas_x_y/mnq_Xy_valid.npz"
ruta_x_y_test = f"{drive_path}/ventanas_x_y/mnq_Xy_test.npz"


#### Para X_train e y_train

In [16]:
if not os.path.exists(ruta_x_y_train):
    print('El archivo no existe -> Generando X_train e y_train: ')
    X_train, y_train = generar_ventanas(mnq_train, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_train, X=X_train, y=y_train)
    print("Guardado:", ruta_x_y_train)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_train)
    data_train = np.load(ruta_x_y_train)
    X_train, y_train = data_train["X"], data_train["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_train.npz


In [17]:
print(f"X_train info: {X_train.shape[0]} ventanas aplanadas en {X_train.shape[1]} números, el equivalente a la window size ({window_size}) por la cantidad de features ({len(features)})")
print(f"y_train info: {y_train.shape[0]} valores que corresponden al retorno a 30 minutos ({target_column})")

X_train info: 276017 ventanas aplanadas en 1260 números, el equivalente a la window size (60) por la cantidad de features (21)
y_train info: 276017 valores que corresponden al retorno a 30 minutos (target_return_30)


#### Para X_valid e y_valid

In [18]:
if not os.path.exists(ruta_x_y_valid):
    print('El archivo no existe -> Generando X_valid e y_valid: ')
    X_valid, y_valid = generar_ventanas(mnq_valid, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_valid, X=X_valid, y=y_valid)
    print("Guardado:", ruta_x_y_valid)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_valid)
    data_valid = np.load(ruta_x_y_valid)
    X_valid, y_valid = data_valid["X"], data_valid["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_valid.npz


#### Para X_test e y_test

In [19]:
if not os.path.exists(ruta_x_y_test):
    print('El archivo no existe -> Generando X_test e y_test: ')
    X_test, y_test = generar_ventanas(mnq_test, features, target_column, window_size)
    np.savez_compressed(ruta_x_y_test, X=X_test, y=y_test)
    print("Guardado:", ruta_x_y_test)
else:
    print("Ya existe -> Cargando desde disco:", ruta_x_y_test)
    data_test = np.load(ruta_x_y_test)
    X_test, y_test = data_test["X"], data_test["y"]

Ya existe -> Cargando desde disco: /content/drive/MyDrive/neural_profit/ventanas_x_y/mnq_Xy_test.npz


## 3. Escalado completo de los X_* e y_*

In [20]:
X_train.shape

(276017, 1260)

In [21]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

In [22]:
def _choose_scaler(scaler_type="standard"):
    st = scaler_type.lower()
    if st in ["standard", "z", "zscore"]:
        return StandardScaler()
    elif st in ["minmax", "min_max"]:
        return MinMaxScaler()
    else:
        raise ValueError("scaler_type debe ser 'standard' o 'minmax'")


In [23]:
def _fit_on_3d(X_train_3d, scaler):
    n, W, F = X_train_3d.shape
    scaler.fit(X_train_3d.reshape(-1, F))
    return scaler

def _transform_3d(X_3d, scaler):
    n, W, F = X_3d.shape
    Xf = X_3d.reshape(-1, F)
    Xs = scaler.transform(Xf).reshape(n, W, F)
    return Xs


In [24]:
def scale_and_save(
    X_train,
    X_valid=None,
    X_test=None,
    scaler_type="standard",
    scaler_path=f"{drive_path}/global_scaler.pkl",
    window_size=None,   # si X_* están en 2D (n, W*F), pasá W para escalar por feature
    verbose=True
):
    """
    Escala X_train (y opcionalmente valid/test) y guarda el escalador.
    - Si X_* es 3D: (n, W, F) -> fit por feature.
    - Si X_* es 2D: (n, W*F). Si pasás window_size=W, reescala por feature reconstruyendo 3D; si no, escala columnas tal cual.

    Return:
        X_train_scaled, X_valid_scaled (o None), X_test_scaled (o None), scaler
    """
    scaler = _choose_scaler(scaler_type)

    # Detectar dimensiones y preparar para fit / transform
    if X_train.ndim == 3:
        # 3D directo
        scaler = _fit_on_3d(X_train, scaler)
        X_train_s = _transform_3d(X_train, scaler)
        X_valid_s = _transform_3d(X_valid, scaler) if X_valid is not None else None
        X_test_s  = _transform_3d(X_test,  scaler) if X_test  is not None else None

    elif X_train.ndim == 2:
        n, tot = X_train.shape
        if window_size is not None:
            # Reescalar por feature: reconstruyo 3D -> escalo -> vuelvo a 2D
            assert tot % window_size == 0, "total de columnas no divisible por window_size"
            F = tot // window_size

            def to3d(X2d):
                return X2d.reshape(X2d.shape[0], window_size, F)

            Xtr3 = to3d(X_train)
            scaler = _fit_on_3d(Xtr3, scaler)

            X_train_s = _transform_3d(Xtr3, scaler).reshape(n, tot)

            if X_valid is not None:
                Xva3 = to3d(X_valid)
                X_valid_s = _transform_3d(Xva3, scaler).reshape(X_valid.shape[0], tot)
            else:
                X_valid_s = None

            if X_test is not None:
                Xte3 = to3d(X_test)
                X_test_s = _transform_3d(Xte3, scaler).reshape(X_test.shape[0], tot)
            else:
                X_test_s = None
        else:
            # Escalado columna a columna (no reconstruye 3D)
            scaler.fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_valid_s = scaler.transform(X_valid) if X_valid is not None else None
            X_test_s  = scaler.transform(X_test)  if X_test  is not None else None
    else:
        raise ValueError("X_train debe ser 2D o 3D.")

    # Guardar escalador
    joblib.dump(scaler, scaler_path)
    if verbose:
        print(f"✅ Scaler guardado en: {scaler_path}")
        print("Shapes escaladas:",
              "X_train", X_train_s.shape,
              "| X_valid", None if X_valid is None else X_valid_s.shape,
              "| X_test",  None if X_test  is None  else X_test_s.shape)

    return X_train_s, X_valid_s, X_test_s, scaler

In [ ]:
# X_train, X_valid, X_test con shape (n, 1260)  # 60 * 21
X_train_s, X_valid_s, X_test_s, scaler = scale_and_save(
    X_train, X_valid, X_test,
    scaler_type="standard",
    scaler_path=f"{drive_path}/global_scaler.pkl",
    window_size=60,   # importante para reescalar por feature
    verbose=True
)

In [ ]:
# Subcarpeta donde querés guardar
save_dir_s = f"{drive_path}/ventanas_x_y_scaled"

# Crear carpeta si no existe
os.makedirs(save_dir_s, exist_ok=True)

In [2]:
#Ruta de x_y escalados
ruta_x_y_train_s = f"{drive_path}/ventanas_x_y_scaled/mnq_Xy_train_scaled.npz"
ruta_x_y_valid_s = f"{drive_path}/ventanas_x_y_scaled/mnq_Xy_valid_scaled.npz"
ruta_x_y_test_s = f"{drive_path}/ventanas_x_y_scaled/mnq_Xy_test_scaled.npz"

NameError: name 'drive_path' is not defined

In [3]:
def save_or_load_scaled_data(X_train_s, X_valid_s, X_test_s, y_train, y_valid, y_test):

  if not os.path.exists(ruta_x_y_train_s):
      print('El archivo no existe -> Guardando X_train_s')
      np.savez_compressed(ruta_x_y_train_s, X=X_train_s, y=y_train)
      print("Guardado:", ruta_x_y_train_s)
  else:
      print("Ya existe -> Cargando desde disco:", ruta_x_y_train_s)
      data_train_s = np.load(ruta_x_y_train_s)
      X_train_s, y_train = data_train_s["X"], data_train_s["y"]

  if not os.path.exists(ruta_x_y_valid_s):
      print('El archivo no existe -> Guardando X_valid_s')
      np.savez_compressed(ruta_x_y_valid_s, X=X_valid_s, y=y_valid)
      print("Guardado:", ruta_x_y_valid_s)
  else:
      print("Ya existe -> Cargando desde disco:", ruta_x_y_valid_s)
      data_valid_s = np.load(ruta_x_y_valid_s)
      X_valid_s, y_valid = data_valid_s["X"], data_valid_s["y"]

  if not os.path.exists(ruta_x_y_test_s):
      print('El archivo no existe -> Guardando X_test_s')
      np.savez_compressed(ruta_x_y_test_s, X=X_test_s, y=y_test)
      print("Guardado:", ruta_x_y_test_s)
  else:
      print("Ya existe -> Cargando desde disco:", ruta_x_y_test_s)
      data_test_s = np.load(ruta_x_y_test_s)
      X_test_s, y_test = data_test_s["X"], data_test_s["y"]

In [4]:
save_or_load_scaled_data

<function __main__.save_or_load_scaled_data(X_train_s, X_valid_s, X_test_s, y_train, y_valid, y_test)>

El archivo no existe -> Guardando X_test_s
Guardado: /content/drive/MyDrive/neural_profit/ventanas_x_y_scaled/mnq_Xy_test_scaled.npz


## 4. Selección y filtrado de features

Función para filtrar features de los X_*

In [26]:
def filter_features_to_model(X_train, X_valid, X_test, features, idx_features_selected, window_size=60):
    """
    Filtra features de los conjuntos X_* manteniendo solo los índices indicados.

    Parámetros:
    -----------
    X_train, X_valid, X_test : np.ndarray
        Arrays en 2D (n_muestras, window_size * n_features) aplanados.
    features : list[str]
        Lista completa de nombres de features en el orden original.
    window_size : int
        Tamaño de la ventana usada en la construcción (ej: 60).
    keep_idx : list[int]
        Lista de índices de features que se quieren conservar (ej: [0,1,2,3,4,7]).

    Retorna:
    --------
    X_train_f, X_valid_f, X_test_f : np.ndarray
        Arrays filtrados en 2D (n_muestras, window_size * n_features_seleccionados).
    features_f : list[str]
        Lista de features seleccionados.
    """

    n_features = len(features)

    # Verificación rápida
    n_total_cols = window_size * n_features
    assert X_train.shape[1] == n_total_cols, "X_train no coincide con window_size * n_features"

    # Calcular columnas a mantener
    cols_to_keep = []
    for idx in idx_features_selected:
        start = idx * window_size
        end = (idx + 1) * window_size
        cols_to_keep.extend(range(start, end))

    # Filtrar arrays
    X_train_f = X_train[:, cols_to_keep]
    X_valid_f = X_valid[:, cols_to_keep]
    X_test_f  = X_test[:, cols_to_keep]

    # Features filtrados
    features_f = [features[i] for i in idx_features_selected]

    print(f"Features seleccionados ({len(features_f)}): {features_f}")
    print("X_train_f shape:", X_train_f.shape)
    print("X_valid_f shape:", X_valid_f.shape)
    print("X_test_f shape :", X_test_f.shape)

    return X_train_f, X_valid_f, X_test_f, features_f

In [27]:
# Crear DataFrame con índice y nombre del feature
tabla_features = pd.DataFrame({"Feature": features})
tabla_features

,Feature
0,open
1,high
2,low
3,close
4,volume
5,momentum_3
6,momentum_10
7,roc_5
8,roc_20
9,rsi_3


In [28]:
# ['open', 'high', 'low', 'close', 'volume',  # OHLCV
 #'momentum_3', 'momentum_10', 'roc_5', 'rsi_3', 'rsi_14', 'stoch_k_20', 'price_ema30',  'atr_norm',
 #'reversal_momentum_factor',  'reversion_vol_momentum_factor', 'factor30']


In [29]:
def get_feature_indices(tabla_features, feature_names):
    """
    Devuelve los índices de los features dados sus nombres.

    Parámetros:
    -----------
    tabla_features : pd.DataFrame
        DataFrame con columnas ["Índice", "Feature"].
    feature_names : list[str]
        Lista de nombres de features a buscar.

    Retorna:
    --------
    list[int] : índices correspondientes a los nombres.
    """
    idx_list = [0, 1, 2, 3, 4] #OHCLV incluido
    for name in feature_names:
        fila = tabla_features.loc[tabla_features["Feature"] == name, "Índice"]
        if not fila.empty:
            idx_list.append(int(fila.values[0]))
        else:
            print(f"⚠️ Feature '{name}' no encontrado en tabla_features.")
    # Ordenar y eliminar duplicados
    idx_list = sorted(set(idx_list))
    return idx_list

In [30]:
# Ejemplo de tabla_features
tabla_features = pd.DataFrame({
    "Índice": list(range(len(features))),
    "Feature": features
})


In [31]:
feature_selected_name = ['open', 'high', 'low', 'close', 'volume',  # OHLCV
 'momentum_3', 'momentum_10', 'roc_5', 'rsi_3', 'rsi_14', 'stoch_k_20', 'price_ema30',  'atr_norm',
 'reversal_momentum_factor',  'reversion_vol_momentum_factor', 'factor30']



In [32]:
 features_select = ['factor30', 'rsi_14', 'price_ema30', 'stoch_k_20', 'bb_percent_30_20', 'reversal_momentum_factor', 'rsi_7', 'reversal_media_factor', 'rsi_3', 'bb_percent_20_15']

In [33]:
# Obtener índices
indices = get_feature_indices(tabla_features, features_select)
print("Índices encontrados:", indices)

Índices encontrados: [0, 1, 2, 3, 4, 9, 10, 11, 12, 13, 14, 15, 17, 18, 20]


In [34]:
import numpy as np

def filter_features_by_index(X_train, X_valid, X_test, features, window_size, keep_idx):
    """
    Filtra features de los conjuntos X_* manteniendo solo los índices indicados.

    Parámetros:
    -----------
    X_train, X_valid, X_test : np.ndarray
        Arrays en 2D (n_muestras, window_size * n_features) aplanados.
    features : list[str]
        Lista completa de nombres de features en el orden original.
    window_size : int
        Tamaño de la ventana usada en la construcción (ej: 60).
    keep_idx : list[int]
        Lista de índices de features que se quieren conservar (ej: [0,1,2,3,4,7]).

    Retorna:
    --------
    X_train_f, X_valid_f, X_test_f : np.ndarray
        Arrays filtrados en 2D (n_muestras, window_size * n_features_seleccionados).
    features_f : list[str]
        Lista de features seleccionados.
    """

    n_features = len(features)

    # Verificación rápida
    n_total_cols = window_size * n_features
    assert X_train.shape[1] == n_total_cols, "X_train no coincide con window_size * n_features"

    # Calcular columnas a mantener
    cols_to_keep = []
    for idx in keep_idx:
        start = idx * window_size
        end = (idx + 1) * window_size
        cols_to_keep.extend(range(start, end))

    # Filtrar arrays
    X_train_f = X_train[:, cols_to_keep]
    X_valid_f = X_valid[:, cols_to_keep]
    X_test_f  = X_test[:, cols_to_keep]

    # Features filtrados
    features_f = [features[i] for i in keep_idx]

    print(f"✅ Features seleccionados ({len(features_f)}): {features_f}")
    print("X_train shape:", X_train_f.shape)
    print("X_valid shape:", X_valid_f.shape)
    print("X_test shape :", X_test_f.shape)

    return X_train_f, X_valid_f, X_test_f, features_f

In [35]:
# Supongamos que querés mantener los features con índices: 0–4, 7, 8, 12, 16
features_to_model = [0,1,2,3,4,7,8,12,16]

X_train_f, X_valid_f, X_test_f, features_f = filter_features_by_index(
    X_train, X_valid, X_test,
    features=features,          # tu lista de 21 features
    window_size=60,
    keep_idx=indices
)

✅ Features seleccionados (15): ['open', 'high', 'low', 'close', 'volume', 'rsi_3', 'rsi_7', 'rsi_14', 'stoch_k_20', 'bb_percent_20_15', 'bb_percent_30_20', 'price_ema30', 'reversal_momentum_factor', 'reversal_media_factor', 'factor30']
X_train shape: (276017, 900)
X_valid shape: (59297, 900)
X_test shape : (59297, 900)


In [36]:
X_train_f.shape

(276017, 900)